# Mini-GPT on Google Colab (GPU)

Train the decoder-only Transformer from the **Mini-GPT** repo on a free Colab GPU.

Choose your dataset in **Section 3**:
- `MODE = 'shakespeare'` -> classic Tiny Shakespeare (English)
- `MODE = 'hinglish'`    -> Hindi-English code-mixed (Roman script)

**Steps:** set the runtime to GPU (**Runtime -> Change runtime type -> GPU**), then **Runtime -> Run all**.

Every cell is safe to re-run: each one resets to `/content` first, so you will never get nested `Mini-GPT/Mini-GPT` folders.

## 1. Check the GPU
Confirm a GPU is attached. If this shows nothing, set the runtime to GPU first.

In [ ]:
!nvidia-smi

## 2. Get the code (fresh clone)
Always starts from `/content` and removes any old copy, so re-running never nests folders. After this, the working directory is `/content/Mini-GPT`.

In [ ]:
%cd /content
!rm -rf Mini-GPT
!git clone https://github.com/shubham-KMX/Mini-GPT.git
%cd /content/Mini-GPT
print('\nFiles in repo:')
!ls

## 3. Choose the dataset
Set `MODE` to either `'shakespeare'` or `'hinglish'`, then run every cell below.

In [ ]:
MODE = 'hinglish'   # 'shakespeare' or 'hinglish'
print('Selected mode:', MODE)

## 4. Prepare the data
- **shakespeare:** nothing to download here -- `train.py` fetches Tiny Shakespeare into `input.txt` automatically.
- **hinglish:** streams a Hindi-English (Roman script) corpus, cleans it, and writes ~5 MB to `input.txt` (overwriting Shakespeare).

> If the default dataset id fails to load, edit the cell to add `--dataset <hf_dataset_id>` with a working Hinglish text dataset.

In [ ]:
%cd /content/Mini-GPT
if MODE == 'hinglish':
    !pip install -q datasets
    !python prepare_hinglish.py --target-mb 5
    print('\n--- First 500 characters of input.txt ---')
    print(open('input.txt', encoding='utf-8').read()[:500])
else:
    print("Shakespeare mode: train.py will download input.txt automatically.")

## 5. Train
`train.py` auto-detects CUDA and reads `input.txt`. This config (256-dim, 6 layers, 6 heads, context 256) trains in a few minutes on a T4 and reaches a validation loss around 1.5.

In [ ]:
%cd /content/Mini-GPT
!python train.py \
  --batch-size 64 \
  --block-size 256 \
  --n-embd 256 \
  --n-head 6 \
  --n-layer 6 \
  --dropout 0.2 \
  --learning-rate 3e-4 \
  --max-iters 5000 \
  --eval-interval 500 \
  --gen-tokens 1000

## 6. Generate from your own prompt
`train.py` saved `mini_gpt.pt`. Reload it and sample without retraining. Change `prompt` to anything.

- Shakespeare model: `ROMEO:`, `KING:`
- Hinglish model: `Aaj kal`, `Mujhe lagta hai`

In [ ]:
%cd /content/Mini-GPT
import torch
from model import GPTLanguageModel

device = 'cuda' if torch.cuda.is_available() else 'cpu'
ckpt = torch.load('mini_gpt.pt', map_location=device)
cfg = ckpt['config']
chars = ckpt['vocab']

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
decode = lambda ids: ''.join(itos[i] for i in ids)

def encode(s):
    return [stoi[c] for c in s if c in stoi]  # skip unknown characters

model = GPTLanguageModel(
    vocab_size=len(chars),
    n_embd=cfg['n_embd'],
    block_size=cfg['block_size'],
    n_layer=cfg['n_layer'],
    n_head=cfg['n_head'],
    dropout=cfg['dropout'],
).to(device)
model.load_state_dict(ckpt['model_state'])
model.eval()

prompt = 'Aaj kal'  # <-- change this
ids = encode(prompt) or encode('\n')
context = torch.tensor([ids], dtype=torch.long, device=device)
out = model.generate(context, max_new_tokens=1000)
print(decode(out[0].tolist()))

## 7. (Optional) Download the checkpoint
Save the trained weights so you can run `generate.py` locally.

In [ ]:
%cd /content/Mini-GPT
from google.colab import files
files.download('mini_gpt.pt')